# Trabajo Práctico 2 - Grupo 02

### Modelo Random Forest

Integrantes:

*   Bermudez, Agustin
*   Calderón, Tiago
*   Gonzalez Pautaso, Mateo
*   Moreyra, Santiago
*   Nieves, Maylen

El objetivo es entrenar un modelo de Random Forest sobre dos representaciones de texto (TF-IDF y BoW) con búsqueda de hiperparámetros, comparando su desempeño contra el baseline de Naive Bayes (~0.66 F1-macro en Kaggle).

Random Forest es un ensemble de árboles de decisión entrenados sobre subconjuntos aleatorios del data y de las features. Al promediar muchos árboles distintos reduce la varianza y generaliza mejor que un árbol solo.

## Importación e instalación de dependencias


In [ ]:
!pip install -q spacy
!python -m spacy download es_core_news_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.9/12.9 MB 72.1 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('es_core_news_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.model_selection import RandomizedSearchCV
from sklearn.model_selection import GridSearchCV

In [ ]:
from common.preprocessing import clean_classical
from common.data_utils import get_split, make_tfidf, make_bow, SEED
from common.evaluation import evaluate
from common.io_utils import save_model, load_model, make_submission

np.random.seed(SEED)

## Carga de datos y preprocesamiento

In [ ]:
train = pd.read_csv("/content/train.csv")
test  = pd.read_csv("/content/test.csv")
X_train_raw, X_val_raw, y_train, y_val = get_split(train)

In [ ]:
print("Preprocesando")
X_train = np.array([clean_classical(t) for t in X_train_raw])
X_val   = np.array([clean_classical(t) for t in X_val_raw])
X_test  = np.array([clean_classical(t) for t in test['text'].values])
print(f"Train: {len(X_train):,} | Val: {len(X_val):,} | Test: {len(X_test):,}")

Preprocesando
Train: 40,800 | Val: 10,200 | Test: 8,500


## Entrega NX: Random Forest + ...

In [ ]:
# Agrupa vectorizador y modelo en un solo objeto
pipe = Pipeline([
    ("tfidf", make_tfidf()),
    ("rf",    RandomForestClassifier(n_estimators=100, class_weight="balanced", random_state=SEED, n_jobs=-1))])

pipe.fit(X_train, y_train)
y_pred = pipe.predict(X_val)
evaluate("rf_tfidf_v1", y_val, y_pred, hyperparams={"n_estimators": 100, "vectorizer": "TF-IDF", "class_weight": "balanced"})

# Reentrenar en train completo y generar submission
pipe.fit(np.concatenate([X_train, X_val]), np.concatenate([y_train, y_val]))
save_model(pipe, "rf_tfidf_v1")
make_submission(pipe, pd.DataFrame({"id": test["id"], "text": X_test}), "submission_rf_tfidf_v1.csv");


=== rf_tfidf_v1 ===
Hiperparámetros: {'n_estimators': 100, 'vectorizer': 'TF-IDF', 'class_weight': 'balanced'}

F1-macro:  0.6061
Precision: 0.6513
Recall:    0.6187
Accuracy:  0.7061

              precision    recall  f1-score   support

    negativa     0.6969    0.8542    0.7675      4080
      neutra     0.5082    0.1819    0.2679      2040
    positiva     0.7487    0.8201    0.7828      4080

    accuracy                         0.7061     10200
   macro avg     0.6513    0.6187    0.6061     10200
weighted avg     0.6799    0.7061    0.6737     10200

Matriz de confusión (filas=real, cols=predicho):
          negativa  neutra  positiva
negativa      3485     197       398
neutra         944     371       725
positiva       572     162      3346
Modelo guardado → models/rf_tfidf_v1.joblib
Predicción guardada → submissions/submission_rf_tfidf_v1.csv  (8500 predicciones)
Distribución: clase 0: 48.8%, clase 1: 7.6%, clase 2: 43.6%
